# Preparar pp_reset con pulsos (prueba del tiempo acumulativo)

Crea **un estado `sp_set` que ya percola** y lo lanza directamente desde **PP_reset** con un
tren de pulsos parametrizable, para comprobar el tiempo acumulativo en la recombinación.

Usa carpetas propias (`Init_data_pulsos_reset/`, `Results_pulsos_reset/`) para **no tocar** `Init_data/` ni `Results/`.

```
parámetros → init → estado sp_set percolante → exec --start-from pp_reset --stop-at pp_reset
```

In [ ]:
import random
import subprocess
import sys
from pathlib import Path

import numpy as np

from RRAM import simulation_config, Percolation
from RRAM.Generation import contar_vecinos_zona_filamento
from RRAM.init_simulation import load_simulation_config
from RRAM.persistence import save_phase_state
from RRAM.voltage_controller import constante, previsualizar

ruta_raiz = Path.cwd()
print('Ruta raíz del proyecto:', ruta_raiz)

## 1 · Parámetros de la prueba

In [ ]:
# ── Carpetas propias de esta prueba ──────────────────────────────────────
init_data_dir = Path('Init_data_pulsos_reset')
results_dir = 'Results_pulsos_reset'

# ── Estado inicial ───────────────────────────────────────────────────────
densidad_vacantes = 14  # vacantes / nm²
grosor_filamento = [2]

# ── Tren de pulsos de PP_reset ───────────────────────────────────────────
# Los umbrales de generación de oxígeno en PP_reset comparan |V| con
# voltaje_gen_oxigeno_pp_1 / pp_2 (1.1 y 1.15 V por defecto): con |V| menor no hay oxígenos.
# (V_on [V], pasos_on, pasos_off, n_pulsos). Cada pulso es off (0 V) → on (V_on).
pulsos = (-1.2, 50, 20, 5)

# ── Ejecución ────────────────────────────────────────────────────────────
seed = None  # None → aleatorio; int → reproducible (fija la semilla de las fases)
usar_muro = False

## 2 · Protocolo de voltaje y CSV → Init_data

In [ ]:
def tren(V, pasos_on, pasos_off, n):
    """n pulsos que empiezan en 0 V: off (0 V) → on (V); al final, un off para acabar en 0 V."""
    return [constante(0.0, pasos_off), constante(V, pasos_on)] * n + [constante(0.0, pasos_off)]


protocolo = {
    "pp_set": [constante(0.0, 1)],  # no se ejecutan: el estado sp_set se construye a mano
    "sp_set": [constante(0.0, 1)],
    "pp_reset": tren(*pulsos),
    "sp_reset": [constante(0.0, 1)],
}

manager = simulation_config.ConfigManager()
manager.add_sweep(sweep_params={
    "densidad_vacantes": [densidad_vacantes],
    "grosor_filamento": [grosor_filamento],
    "protocolo_voltaje": [protocolo],
})
manager.export_to_init_data(str(init_data_dir))
num_simulaciones = len(manager.simulations)
print(f'Generadas {num_simulaciones} configuraciones en {init_data_dir}/.')

In [ ]:
# Previsualización del voltaje (sin simular)
previsualizar(manager)

## 3 · INIT — estados iniciales

In [ ]:
resultado = subprocess.run(
    [sys.executable, '-m', 'RRAM', 'init', '--init-data-dir', str(init_data_dir)],
    cwd=str(ruta_raiz),
    capture_output=True, text=True,
)
print(resultado.stdout)
if resultado.returncode != 0:
    print('STDERR:', resultado.stderr)
    raise RuntimeError('init falló')

## 4 · Estado `sp_set` que percola

Se construye en LRS con la densidad indicada, garantizando un camino de percolación (a 0 V y temperatura inicial).

In [ ]:
def crear_estado_inicial_garantizando_percolacion(eje_x, eje_y, num_filamentos, densidad_vacantes, grosores_filamento=None):
    """
    Genera un estado inicial respetando ESTRICTAMENTE la densidad física.
    Si la densidad permite un camino serpenteante, lo crea.
    Si solo permite una fila, genera una línea recta.
    Si es insuficiente para cruzar, el filamento queda interrumpido.
    """
    estado = np.zeros((eje_x, eje_y), dtype=int)
    cte_red_nm = 0.25
    area_celda_nm2 = cte_red_nm**2

    if grosores_filamento is None:
        grosores = [2] * num_filamentos
    elif isinstance(grosores_filamento, (int, np.integer)):
        grosores = [int(grosores_filamento)] * num_filamentos
    else:
        grosores_raw = list(grosores_filamento)
        if len(grosores_raw) < num_filamentos:
            grosores_raw += [grosores_raw[-1]] * (num_filamentos - len(grosores_raw))
        grosores = [int(g) for g in grosores_raw[:num_filamentos]]

    ancho_zona = eje_x // num_filamentos

    for n in range(num_filamentos):
        inicio_rango = n * ancho_zona
        fin_rango = (n + 1) * ancho_zona - 1 if n < num_filamentos - 1 else eje_x - 1
        fila_central = (inicio_rango + fin_rango) // 2
        grosor = grosores[n]

        x_start = max(0, fila_central - grosor)
        x_end = min(eje_x, fila_central + grosor + 1)

        celdas_totales_region = (x_end - x_start) * eje_y
        if celdas_totales_region <= 0:
            continue

        # Presupuesto estricto de vacantes
        num_vacantes_totales = int(round(celdas_totales_region * area_celda_nm2 * densidad_vacantes))
        num_vacantes_totales = min(num_vacantes_totales, celdas_totales_region)

        # Paso A: camino con codos aleatorios solo si sobran vacantes
        pos_x_actual = fila_central
        vacantes_restantes = num_vacantes_totales
        print(f"Filamento {n+1}/{num_filamentos}: presupuesto de vacantes = {num_vacantes_totales}")
        for y in range(eje_y):
            if vacantes_restantes <= 0:
                break

            if estado[pos_x_actual, y] == 0:
                estado[pos_x_actual, y] = 1
                vacantes_restantes -= 1

            if y < eje_y - 1:
                pasos_minimos_restantes = (eje_y - 1) - y
                if vacantes_restantes > pasos_minimos_restantes:
                    nuevo_x = pos_x_actual + random.choice([-1, 0, 1])
                    nuevo_x = max(x_start, min(nuevo_x, x_end - 1))
                    if nuevo_x != pos_x_actual:
                        if estado[nuevo_x, y] == 0:
                            estado[nuevo_x, y] = 1
                            vacantes_restantes -= 1
                        pos_x_actual = nuevo_x

        # Paso B: vacantes sobrantes repartidas al azar en la zona
        if vacantes_restantes > 0:
            celdas_libres_x, celdas_libres_y = np.where(estado[x_start:x_end, :] == 0)
            celdas_libres_x += x_start
            num_libres = len(celdas_libres_x)
            vacantes_a_colocar = min(vacantes_restantes, num_libres)
            indices_elegidos = np.random.choice(num_libres, size=vacantes_a_colocar, replace=False)
            estado[celdas_libres_x[indices_elegidos], celdas_libres_y[indices_elegidos]] = 1

    return estado

In [ ]:
def generar_estado_sp_set(num_simulation, init_data_dir, tiempo_sp_set=0.0):
    """Construye y guarda phase_state_{n+1}_sp_set.*. Devuelve True si el dispositivo percola."""
    cfg = load_simulation_config(num_simulation, init_data_dir=init_data_dir)
    params = cfg.params
    sim_ctes = cfg.sim_ctes

    eje_x = params.y_size  # filas (alto del dispositivo)
    eje_y = params.x_size  # columnas (distancia entre electrodos)

    actual_state = crear_estado_inicial_garantizando_percolacion(
        eje_x=eje_x,
        eje_y=eje_y,
        num_filamentos=int(sim_ctes.num_filamentos),
        densidad_vacantes=params.densidad_vacantes,
        grosores_filamento=sim_ctes.grosor_filamento,
    )

    percola = Percolation.is_path(actual_state.astype(int))

    save_phase_state(
        state_dict={
            "actual_state": actual_state,
            "sim_ctes": sim_ctes,
            "params": params,
            "Temperatura_final": np.full((eje_x, eje_y + 2), float(params.init_temp), dtype=np.float64),
            "percola": percola,
            "centros_calculados": list(cfg.cf_centros),
            "tiempo_sp_set": tiempo_sp_set,
            "vecindad_inicial": contar_vecinos_zona_filamento(actual_state=actual_state, cf_ranges=cfg.cf_ranges),
        },
        phase_name="sp_set",
        num_simulation=num_simulation + 1,
        init_data_dir=init_data_dir,
    )
    return percola, actual_state


for num_sim in range(num_simulaciones):
    percola, estado = generar_estado_sp_set(num_sim, init_data_dir)
    print(f'sim {num_sim} (n_save={num_sim + 1}): {"✓ percola" if percola else "⚠ NO percola"} · vacantes = {int(estado.sum())}')
    if not percola:
        raise RuntimeError('El estado generado no percola; sube la densidad de vacantes.')

## 5 · EXEC — solo PP_reset

In [ ]:
# PP_reset termina con FilamentosNoDestruidosException (código de salida 1) si el protocolo
# no llega a romper el filamento; es esperable en pruebas cortas. Los datos ya están guardados
# antes de esa excepción, pero la excepción guarda una figura y necesita que Figures/ exista.
for num_sim in range(num_simulaciones):
    (Path(results_dir) / f'simulation_{num_sim + 1}' / 'Figures').mkdir(parents=True, exist_ok=True)
    cmd = [sys.executable, '-m', 'RRAM', 'exec', str(num_sim),
           '--init-data-dir', str(init_data_dir),
           '--results-dir', results_dir,
           '--start-from', 'pp_reset', '--stop-at', 'pp_reset']
    if not usar_muro:
        cmd += ['--no-muro']
    if seed is not None:
        cmd += ['--seed', str(seed)]
    print('exec:', ' '.join(cmd[1:]))
    completed = subprocess.run(cmd, cwd=str(ruta_raiz))
    print('código de salida:', completed.returncode)